TRANSFER LEARNING CON KERAS (RESNET, VGG)

Addestrare una rete neurale da zero è costoso: serveno milioni di dati, GPU potenti, settimane di trainin, competenze alte, con il Transfer Learning prendi un modello già addestrato da Google, Meta, Microsoft, ecc su milioni di immagini e lo 'riusi'
Il modello ha già imparato forme, bordi, texture, oggetti, tu gli insegni solo il tuo problema specifico.
Esempi ERP: riconoscimenti visivo difetti su pezzi meccanici, classificazione documenti, riconoscimento palle o colli, OCR avanzato, riconoscimento immagini in magazzino.
In azienda quasi nessuno allena da zero, si usa transfer learning.

Una CNN classica funziona così:
immagine - > estrazione caratteristiche - > classificazione
Le prime parti della rete imparano:
- linee
- curve
- texture
- forme
Le ultime:
- questo è un gatto
- questo è un cane
Nel transfer learning:
- tieni le parti generiche
- cambi solo la testa finale.
immagine - > rete preaddestrata - > tua classificatore
I modelli più famosi sono:
- Keras
- TensorFlow
- Transfer Learning
- Convolutional Neural Network
I modelli pre-trained più usati:
- ResNet (sviluppato da Microsoft, molto profondo, introduce le residual connections, oggi è uno standard industriale)
- VGG (più vecchio, architettura semplice, ottimo da capire, pesante e lento rispetto a ResNet)
- EfficientNet (oggi molto usato, ottimo rapporto qualità/prestazioni)
- MobileNet (pensato per cellulari e dispositivi embedded)

CARICAMENTO MODELLO PRETRAINED
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)
weights='imagenet' usa pesi già allenati su ImageNet
ImageNet include milioni di immagini, migliaia di categorie
Include_top=False (togliamo il classificatore finale originale)

BLOCCARE I LAYER

base_model.trainable = False

Importante perchè non usiao il modello con i dati che indicano la conoscenza del modello.
Non tocchi la conoscenza del modello, usi il metodo solo me estrattore caratteristiche.

AGGIUNGERE UN CLASSIFICATORE

from tensorflow.keras import layers, models

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

stai dicendo: usa ResNet per estrarre caratteristiche, poi classifico io.

COMPILE

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

TRAINING

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

alleni solo gli ultimi layer

for layer in base_model.layers[:-20]:
    layer.trainable = False

Invece di addestrare una rete nel vuoto si può farla salire sulla spalle dei giganti, sfruttando modelli che hanno gà visto milioni di immagini.

- Il valore della conoscenza pre-addestrata (perchè utilizzare i pesi è meglio che invetarli
- Strategia operative
- Pratica con VGG26: per vedere come si manipola l'intelligenza già esistente

Nel Deep Learnig, in particolare nel Transfer learning, segue la logica di condivisione dei modelli
Perchè dobbiamo sprecare milioni di dati e di tempo per capire colori e linee,

I primi layer di una rete convoluzionale imparano a riconoscere forme universali, come linee, angoli, txture, che sono utili per quasi ogni compito di visione artificiale.
- Utilizzare pesi già ottimizzati permette di evitare la fase critica di inizzializzazione casuale, portando la rete a convergenza molto più velovemente.
- Questa tecnica è la chiave per ottenere risultati eccellenti anche quando disponiamo di set di dati molto piccoli, dove un addestramento da zero potrebbe inevitabilmente all'overfitting.


Da dove arriva tutta questa 'saggezza' che andiamo ad ereditare?
IMAGE NET
La maggior parte dei modelli pre-addestrati è stata istruita su oltre un milioni di immagini appartenenti a mille categorie diverse, creando una base di conoscenza visiva estramamente robusta.
Le caratteristiche estratte dei layer profondi, diventano sempre più specifiche, ma i layer iniziali agiscono come filtri visivi generici validi per ogni immagine medica/satellitare/neurale.
Riutilizzare modelli esistenti risparmia giorni di calcolo su cluster di GPU costosi, rendendo l'intelligenza 

Gerarchia Visiva
dal generico allo specifico
Una rete neurale profonda è come un sistema di filtri progressivi. All'inizio vede solo contrasti e colori; a metà riconosce pattern geometrici, alla fine identifica, ruote o petali
Il transfer Learning ci permette di mantenere intatti i 'sensori' visivi già addestrati e di sostituire solo la logica decisionale finale che interpreta quegli stimoli (sostituiamo solo la testa del modello)

Quanta libertà dobbiao dare alla rete di cambiare idea?
Gerarchia Visiva
Dal generico allo specifico.
Una rete neurale è come un sistema di filtri progressivi. All'inizio vede solo contrasti e colori; a metà riconosce pattern geometrici; alla fine identifica occhi, ruote, o petali.
Il Transfer Learning ci permette di mantenere intatti i 'sensori' visivi già addestrati e di sostituire solo la loica decisionale finale che interpreta quegli stimoli.

Feature Extractione vs Fine-Tuning
Scegliere la profondità dell'intervento.
Esistono due modi principali per applicare il Transfer Learning. Possiamo usare il modello pre-addestrato come un blocco fisso o possiamo permettergli di adattarsi leggermente al nostro nuovo compito
Capire quando fermarsi alla sola estrazione delle caratteristiche o quando procedere con un raffinamento dei pesi è fondamentale per bilanciare  precisione e stabiltà del modello.


Strategia di adattamento
- Nella Feature Extraction manteniamo i pesi del modello base congelati e addestriamo solo un nuovo classificatore aggiunto in cima alla rete
- Il Fine Tuning consiste nello sbloccare alcuni degli ultimi layer del modello base per permettere loro di specializzarsi sui dettagli specifici del nuovo dataset.
- Il congelamento dei layer si ottiene impostando la proprietà 'trainable' a false, impedendo all'ottimizzatore di modificare i pesi durante la backpropagation.
- L'aggiornamento dei pesi durante il fine-tuning deve avvenire con un tasso di apprendimento molto basso per non distruggere la conoscenza preesistente.
Nella feature extraction congeliamo solo il corpo ed estraiamo solo la testa il classificatore.
Nel fine-tuning sblocchiamo solo gli ultimi layer, per permettere loro di specializzarsi

Quando usare cosa
- Se abbiamo un dataset piccolo usiamo la feature extraction, è l'unica via sicura per evitare che la rete impari a memoria il tuo piccolo set di dati.
- Con una buona quantità di dati il fine-tuning permette di spremere l'ultimo punto percentuale di accuratezza adattando i filtri di alto livello al tuo dominio specifico.
- Se le immagini a nostra disposizione sono molto diverse da ImageNet (es. scansioni radiografiche) il Fine-Tunning diventa più necessario per adattare la visone della rete.
C'è però un pericolo catastrofico nella sbloccare questi ultimi layer: il rishio del Catastrophic Forgetting

C'è pero' un perisolo nel vedere tutti gli animali. 
E' come cercare di insegnare una nuova lingua a qualcuno cancellandogli la memoria: il risultato sarebbe un disastro. Il segreto è addestrare prima la parte nuova e poi, con cautela, rifinire il resto.
Se sblocchiamo i layer troppo presto (con un'energia troppo alta), rischiamo di cancellare tutta la seggezza della rete

VGG16
VGG16 è un'architettura che ha fatto la storia del DeepLearning per la sua eleganza e potenza. in Keras scaricare questo gigante è banale, 

Configurazione VGG16
- Il parametro include_top=False (comando del chirurgo, stiamo rimovendo la testa della rete.) ci permette di rimuovere i layer densi originali di ImageNet, mantenendo solo la potente base convoluzionale.
weights=imagenet Keras scarica i pesi già ottimizzati pronti per essere utilizzati come estrattori di feature.

Ogni modello preaddestrato ha esigenze specifiche, VGG16 vuole che le immagini siano preprocessate con la sua funzione dedicata preprocess_input, inoltre richiede che all'uscita del modello base, dobbiamo trasformare le mappe 2D in vettori 1D usando layer di Flatten o meglio GlobalAveragePooling2D, per appiattire tutto, che riduce drasticamente il numero di parametri e ci protegge dall'overfitting.
Controllare sempre il summary(), se vedete milioni di parametri totale ma sole pocoche migliaia addestrabili allora vuol dire che si è fatto un ottimo lavoro di congemalento.


Keras è un catalogo importante. Oltre a VGG16 si ha a disposizione molti altri modelli come ResNet, Inception e MobilNet (pensata per lo smartphone). La logica di caricamente e congelamento rimane identica per tutti (carica, congela ed aggiunti la testa) . 
Imparare questa tecnica ci da' accesso immediato allo stato dell'arte della ricerca mondiale, permettendoci di costruire applicazioni professionali in pochi minuti.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# 1. CARICAMENTO DEL MODELLO BASE (Punto chiave 3)
# include_top=False rimuove i layer densi finali (il classificatore a 1000 classi)
vgg_base = applications.VGG16(weights='imagenet', 
                             include_top=False, #eliminiamo i layer finali
                             input_shape=(224, 224, 3)) #shape di input delle immagini

# 2. CONGELAMENTO DEI PESI (Punto chiave 2)
# Impediamo all'ottimizzatore di modificare i pesi già appresi su ImageNet
vgg_base.trainable = False #modello non addestrabili sui suoi pesi, quindi dobbiamo aggiungere i layer successivi finali

# 3. COSTRUZIONE DEL MODELLO COMPLESSIVO
model = models.Sequential([
    vgg_base,                    # La base convoluzionale "intelligente"
    layers.GlobalAveragePooling2D(), # Trasforma le mappe 2D in un vettore 1D (indicato per l utilizzo con VGG16)
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),         # Regolarizzazione per evitare overfitting
    layers.Dense(10, activation='softmax') # Esempio: 10 nuove categorie (deve classificare su 10 categorie)
])

# 4. COMPILAZIONE
# Usiamo Adam con il learning rate standard perché la base è congelata
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Ispezione dei parametri
model.summary() #per visualizzare i parametri di addestramento

print(f"\nLayer del modello base congelati: {len(vgg_base.layers)}")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,586 (56.64 MB)

 Trainable params: 133,898 (523.04 KB)

 Non-trainable params: 14,714,688 (56.13 MB)


Layer del modello base congelati: 19


I parametri addestrati sono circa 14714688 parametri, addestrabili 133898

L'eredità è tutto, risparmiando tempo, campioni di dati e potenza di calcolo.
